In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader


def build_vector_store(file_path):
    """Load a PDF, split it into chunks, embed chunks, and return a FAISS vector store."""
    loader = PyPDFLoader(file_path)
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
    )

    chunks = splitter.split_documents(documents)

    embeddings = OpenAIEmbeddings()
    vector_store = FAISS.from_documents(chunks, embeddings)

    return vector_store

In [2]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent as lc_create_agent
from langchain_core.tools import tool


def create_agent(vector_store):
    """Create and return a LangChain agent with a document search tool."""
    llm = ChatOpenAI(temperature=0)

    @tool("document_search", description="Search enterprise documents")
    def search_docs(query: str) -> str:
        """Search enterprise documents for content relevant to the user query."""
        docs = vector_store.similarity_search(query, k=4)
        if not docs:
            return "No relevant documents found."
        return "\n\n".join(doc.page_content for doc in docs)

    agent = lc_create_agent(
        model=llm,
        tools=[search_docs],
        system_prompt="You are a helpful assistant that searches enterprise documents.",
    )

    return agent

In [5]:
import importlib
from pathlib import Path
from fastapi import FastAPI
from dotenv import load_dotenv

BASE_DIR = Path.cwd()
if not (BASE_DIR / "key.env").exists() and (BASE_DIR / "RAG" / "key.env").exists():
    BASE_DIR = BASE_DIR / "RAG"

load_dotenv(BASE_DIR / "key.env")

try:
    import RAG.RAG_pipeline as RAG_pipeline
    import RAG.agent as agent_module
except ModuleNotFoundError:
    import RAG_pipeline
    import agent as agent_module

importlib.reload(RAG_pipeline)
importlib.reload(agent_module)

build_vector_store = RAG_pipeline.build_vector_store
create_agent = agent_module.create_agent

app = FastAPI()

vector_store = build_vector_store(r"C:\Users\Visha\Downloads\DE\Vishal.pdf")
agent = create_agent(vector_store)


@app.post("/ask")
async def ask_question(query: str):
    result = agent.invoke({"messages": [{"role": "user", "content": query}]})
    response = result["messages"][-1].content
    return {"response": response}

In [30]:
pip install uvicorn fastapi

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: uvicorn in c:\users\visha\appdata\local\programs\python\python310\lib\site-packages (0.41.0)




[notice] A new release of pip available: 22.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
messages = [{"role": "assistant", "content": "Ask me anything from your PDF."}]

while True:
    q = input("You: ").strip()
    if q.lower() in {"exit", "quit"}:
        break

    messages.append({"role": "user", "content": q})
    result = agent.invoke({"messages": messages})
    ans = result["messages"][-1].content
    print("Bot:", ans)
    messages.append({"role": "assistant", "content": ans})
